# 🧪 Sesión 05 · Pydantic v2 / DTOs en PHP y Tipado Estático

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué pasa si el JSON no trae el tipo de dato que esperas?

---

## 🖥️ Así se verá el resultado final

```http
$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: application/json" \
       -d '{"titulo":"Rayuela","anio":"1963","precio":250.5,"disponible":true}'
HTTP/1.1 201 Created
{"id":1,"titulo":"Rayuela","anio":1963,"precio":250.5,"disponible":true}

$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: application/json" \
       -d '{"titulo":"Rayuela","anio":"mil novecientos"}'
HTTP/1.1 422 Unprocessable Entity

$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: application/json" \
       -d '{"titulo":"Rayuela","anio":1963,"color":"rojo"}'
HTTP/1.1 422 Unprocessable Entity
```

_El campo extra también se rechaza: el contrato es estricto, no tolerante._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Definir un DTO con Pydantic v2.
- Explicar la coerción de tipos y el error 422.
- Separar el modelo de entrada del modelo de salida.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Un libro válido se crea con 201 → HTTP 201
- [ ] La cadena numérica se convierte a entero → HTTP 201
- [ ] Un año no numérico produce 422 → HTTP 422
- [ ] Un campo extra produce 422 → HTTP 422
- [ ] El listado devuelve los libros creados → HTTP 200

## 🧩 El reto

Define el contrato de un libro con un DTO de entrada y otro de salida. El servidor decide qué es válido antes de guardar.

**Requisitos del contrato**

- El modelo de entrada exige titulo (str), anio (int), precio (float) y disponible (bool).
- La coerción de "1967" a 1967 debe funcionar; "mil novecientos" debe fallar con 422.
- Los campos desconocidos se rechazan con 422 (extra="forbid").
- La respuesta nunca inventa campos: devuelve el id y los datos del libro.

> **Pista conceptual:** En Pydantic v2, `model_config = ConfigDict(extra="forbid")` convierte el DTO en un contrato cerrado.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion05.md` con tu DTO de entrada y salida.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu DTO aceptará y rechazará exactamente esto")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'Un libro válido se crea con 201',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963, 'precio': 250.5, 'disponible': True},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela', 'anio': 1963, 'precio': 250.5, 'disponible': True}},
 {'descripcion': 'La cadena numérica se convierte a entero',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'La casa verde', 'anio': '1966', 'precio': 180, 'disponible': False},
  'estado': 201,
  'json_contiene': {'id': 2, 'anio': 1966}},
 {'descripcion': 'Un año no numérico produce 422',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 'mil novecientos', 'precio': 250.5, 'disponible': True},
  'estado': 422},
 {'descripcion': 'Un campo extra produce 422',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963, 'precio': 250.5, 'disponible': True, 'color': 'rojo'},
  'estado': 422},
 {'descripcion': 'El listado devuelve los libros creados',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}, {'id': 2}]}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion05.md` a `auditorias/auditoria_sesion05.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
